In [ ]:
#4

import socket
import pandas as pd

# Step 1: Enter target system
target = input("Enter target IP address (default: 127.0.0.1): ")

if not target.strip():
    target = "127.0.0.1"

# Step 2: Define common ports and their services
ports = {
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    139: "NetBIOS",
    143: "IMAP",
    443: "HTTPS",
    445: "SMB",
    3306: "MySQL",
    3389: "RDP",
    5432: "PostgreSQL",
    8080: "HTTP Proxy"
}

# Step 3: Identify potentially risky services
risky_ports = {
    21: "FTP may transmit credentials without encryption",
    23: "Telnet transmits data without encryption",
    445: "SMB may expose file-sharing services",
    3389: "RDP may expose remote desktop access",
    3306: "MySQL database may be exposed",
    5432: "PostgreSQL database may be exposed"
}

results = []

print("\nVULNERABILITY IDENTIFICATION REPORT")
print("-----------------------------------")

# Step 4: Scan common TCP ports
for port, service in ports.items():

    try:
        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        sock.settimeout(0.5)

        status = sock.connect_ex((target, port))

        if status == 0:

            if port in risky_ports:
                risk = "Potential Risk"
                details = risky_ports[port]
            else:
                risk = "Review Required"
                details = "Open service should be reviewed"

            print(f"Port {port} - {service}: OPEN")

        else:
            risk = "No Open Port Detected"
            details = "Service not accessible"

        if status == 0:
            results.append({
                "Port": port,
                "Service": service,
                "Status": "Open",
                "Risk": risk,
                "Details": details
            })

        sock.close()

    except socket.error:
        print(f"Error scanning port {port}")

# Step 5: Display vulnerability report
print("\nSECURITY ASSESSMENT RESULTS")

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))

    # Step 6: Save report
    df.to_csv("vulnerability_report.csv", index=False)

    print("\nReport saved as vulnerability_report.csv")

else:
    print("No open ports detected in the selected port list.")

# Step 7: Security recommendations
print("\nSECURITY RECOMMENDATIONS")

recommendations = [
    "Disable unnecessary open ports.",
    "Use SSH instead of Telnet.",
    "Use secure protocols instead of unencrypted FTP.",
    "Enable firewall protection.",
    "Restrict database access.",
    "Keep operating systems and software updated.",
    "Use strong passwords and MFA."
]

for item in recommendations:
    print("-", item)

print("\nVulnerability assessment completed!")


In [ ]:
!pip install zxcvbn pandas

In [ ]:
#6

import pandas as pd
import re
from zxcvbn import zxcvbn

# Step 1: Sample passwords for testing
passwords = [
    ("Numeric", "12345678"),
    ("Lowercase", "hello"),
    ("Uppercase", "HELLO"),
    ("Alphabetic", "HelloWorld"),
    ("Alphanumeric", "Hello123"),
    ("Special Characters", "@#$%&*!"),
    ("Common Password", "password123"),
    ("Repeated Characters", "aaaaaaa111"),
    ("Sequential Password", "Abc12345"),
    ("Mixed Characters", "Hello@2026"),
    ("Long Passphrase", "Blue-River-Mountain-Sky-Tree"),
    ("Strong Password", "V9@kL2#pQ8!mX5")
]

# Step 2: Password analysis function
def analyze_password(password):

    # Check character types
    length = len(password)
    uppercase = bool(re.search(r"[A-Z]", password))
    lowercase = bool(re.search(r"[a-z]", password))
    digits = bool(re.search(r"[0-9]", password))
    special = bool(re.search(r"[^A-Za-z0-9]", password))

    # Check common and predictable patterns
    common = password.lower() in [
        "password", "password123", "12345678",
        "qwerty", "admin", "welcome"
    ]

    repeated = bool(re.search(r"(.)\1{3,}", password))
    sequential = any(
        sequence in password.lower()
        for sequence in [
            "12345", "23456", "34567",
            "abcde", "qwerty"
        ]
    )

    # Evaluate using zxcvbn
    result = zxcvbn(password)
    score = result["score"]

    # Determine password strength
    if common or repeated or sequential:
        strength = "Very Weak" if score <= 2 else "Moderate"
    else:
        strength = {
            0: "Very Weak",
            1: "Weak",
            2: "Moderate",
            3: "Strong",
            4: "Very Strong"
        }[score]

    # Generate recommendations
    suggestions = []

    if length < 12:
        suggestions.append("Increase password length")

    if not uppercase:
        suggestions.append("Add uppercase letters")

    if not lowercase:
        suggestions.append("Add lowercase letters")

    if not digits:
        suggestions.append("Add numbers")

    if not special:
        suggestions.append("Add special characters")

    if common:
        suggestions.append("Avoid common passwords")

    if repeated:
        suggestions.append("Avoid repeated characters")

    if sequential:
        suggestions.append("Avoid sequential patterns")

    if not suggestions:
        suggestions.append("Maintain a unique password")

    return {
        "Length": length,
        "Uppercase": uppercase,
        "Lowercase": lowercase,
        "Numbers": digits,
        "Special": special,
        "zxcvbn Score": score,
        "Strength": strength,
        "Recommendations": ", ".join(suggestions)
    }


# Step 3: Analyze all sample passwords
results = []

for password_type, password in passwords:

    analysis = analyze_password(password)

    results.append({
        "Password Type": password_type,
        "Password": "*" * len(password),
        **analysis
    })

# Step 4: Display analysis report
df = pd.DataFrame(results)

print("PASSWORD STRENGTH AUDIT REPORT\n")
print(df.to_string(index=False))

# Step 5: Count password strength categories
print("\nPASSWORD STRENGTH SUMMARY")
print(df["Strength"].value_counts())

# Step 6: Save audit report
df.to_csv("password_audit_report.csv", index=False)

print("\nPassword audit report saved successfully!")


In [ ]:
#7
import pyotp
import time
import hashlib

# Generate a secret key
secret = pyotp.random_base32()

# Create TOTP object
totp = pyotp.TOTP(secret)

# User registration
username = input("Enter username: ")
password = input("Create password: ")

# Store password hash
password_hash = hashlib.sha256(
    password.encode()
).hexdigest()

print("\nAccount registered successfully!")

# Login process
print("\n--- USER LOGIN ---")

entered_username = input("Enter username: ")
entered_password = input("Enter password: ")

# Verify username and password
entered_hash = hashlib.sha256(
    entered_password.encode()
).hexdigest()

if (entered_username == username and
        entered_hash == password_hash):

    print("\nPassword verified!")

    # Generate OTP
    otp = totp.now()

    print("Your OTP is:", otp)

    # Verify OTP
    entered_otp = input("Enter OTP: ")

    if totp.verify(entered_otp):
        print("\nLogin Successful!")
        print("MFA Authentication Passed.")
    else:
        print("\nInvalid OTP!")
        print("Access Denied.")

else:
    print("\nInvalid username or password!")
    print("Access Denied.")


In [ ]:
!pip install cryptography

In [ ]:
#8

from cryptography.fernet import Fernet

# Generate AES encryption key
key = Fernet.generate_key()

# Create encryption object
cipher = Fernet(key)

# Input plaintext
message = input("Enter message to encrypt: ")

# Encryption
encrypted_message = cipher.encrypt(message.encode())

print("\nEncrypted Message:")
print(encrypted_message.decode())

# Decryption
decrypted_message = cipher.decrypt(encrypted_message)

print("\nDecrypted Message:")
print(decrypted_message.decode())


In [ ]:
!pip install cryptography

In [ ]:
#9


from cryptography.fernet import Fernet

# Generate AES encryption key
key = Fernet.generate_key()

# Create encryption object
cipher = Fernet(key)

# Input plaintext
message = input("Enter message to encrypt: ")

# Encryption
encrypted_message = cipher.encrypt(message.encode())

print("\nEncrypted Message:")
print(encrypted_message.decode())

# Decryption
decrypted_message = cipher.decrypt(encrypted_message)

print("\nDecrypted Message:")
print(decrypted_message.decode())




In [ ]:
C:\Program Files\OpenSSL-Win64\bin

In [ ]:
# 10
import os

# Step 1: Check OpenSSL
os.system("openssl version")

# Step 2: Generate RSA Private Key
os.system("openssl genpkey -algorithm RSA -out private_key.pem -pkeyopt rsa_keygen_bits:2048")

# Step 3: Generate Public Key
os.system("openssl pkey -in private_key.pem -pubout -out public_key.pem")

# Step 4: Create Message File
with open("message.txt", "w") as file:
    file.write("Hello, this is a digital signature practical.")

print("\nMessage file created successfully!")

# Step 5: Generate Digital Signature
os.system("openssl dgst -sha256 -sign private_key.pem -out signature.bin message.txt")

print("\nDigital signature generated successfully!")

# Step 6: Verify Digital Signature
print("\nOriginal Message Verification:")
os.system("openssl dgst -sha256 -verify public_key.pem -signature signature.bin message.txt")

# Step 7: Modify Message
with open("message.txt", "w") as file:
    file.write("Modified message")

# Step 8: Verify Modified Message
print("\nModified Message Verification:")
os.system("openssl dgst -sha256 -verify public_key.pem -signature signature.bin message.txt")

# Step 9: Display Generated Files
print("\nGenerated Files:")
for file in ["private_key.pem", "public_key.pem",
             "message.txt", "signature.bin"]:
    print(file, ":", "Created" if os.path.exists(file) else "Not Found")


In [ ]:
# 11


import hashlib

# Step 1: Input message
message = input("Enter a message: ")

# Step 2: Generate SHA-256 hash
hash_value = hashlib.sha256(message.encode()).hexdigest()

print("\nGenerated SHA-256 Hash:")
print(hash_value)

# Step 3: Enter message again for verification
verify_message = input("\nEnter message to verify: ")

# Step 4: Generate hash of verification message
verify_hash = hashlib.sha256(
    verify_message.encode()
).hexdigest()

print("\nVerification Hash:")
print(verify_hash)

# Step 5: Compare hash values
if hash_value == verify_hash:
    print("\nHash Verification Successful!")
    print("Data Integrity Maintained.")
else:
    print("\nHash Verification Failed!")
    print("Data has been modified.")


In [ ]:
!pip install cryptography

In [ ]:
#12

from cryptography.fernet import Fernet
import os

# Step 1: Generate encryption key
key = Fernet.generate_key()

# Save key securely
with open("secret.key", "wb") as key_file:
    key_file.write(key)

# Step 2: Create Fernet encryption object
cipher = Fernet(key)

# Step 3: Create a sample file
filename = "sample.txt"

with open(filename, "w") as file:
    file.write("This is a confidential file.")

print("Original file created successfully!")

# Step 4: Read original file
with open(filename, "rb") as file:
    data = file.read()

# Step 5: Encrypt file content
encrypted_data = cipher.encrypt(data)

with open("encrypted_file.txt", "wb") as file:
    file.write(encrypted_data)

print("File encrypted successfully!")

# Step 6: Protected file sharing
print("\nEncrypted file is ready for sharing.")
print("Share encrypted_file.txt securely.")
print("Share secret.key separately with the receiver.")

# Step 7: Load secret key for receiver
with open("secret.key", "rb") as key_file:
    receiver_key = key_file.read()

receiver_cipher = Fernet(receiver_key)

# Step 8: Decrypt received file
with open("encrypted_file.txt", "rb") as file:
    encrypted_content = file.read()

decrypted_data = receiver_cipher.decrypt(encrypted_content)

with open("decrypted_file.txt", "wb") as file:
    file.write(decrypted_data)

print("\nFile decrypted successfully!")

# Step 9: Display decrypted content
with open("decrypted_file.txt", "r") as file:
    print("\nDecrypted File Content:")
    print(file.read())

# Step 10: Display generated files
print("\nGenerated Files:")
for name in ["secret.key", "sample.txt",
             "encrypted_file.txt", "decrypted_file.txt"]:
    print(name, ":", "Created" if os.path.exists(name) else "Not Found")


In [ ]:
!pip install cryptography

In [ ]:
#14

from cryptography.fernet import Fernet
import os

# Step 1: Generate encryption key
key = Fernet.generate_key()

# Save key securely
with open("secret.key", "wb") as key_file:
    key_file.write(key)

# Step 2: Create Fernet encryption object
cipher = Fernet(key)

# Step 3: Create a sample file
filename = "sample.txt"

with open(filename, "w") as file:
    file.write("This is a confidential file.")

print("Original file created successfully!")

# Step 4: Read original file
with open(filename, "rb") as file:
    data = file.read()

# Step 5: Encrypt file content
encrypted_data = cipher.encrypt(data)

with open("encrypted_file.txt", "wb") as file:
    file.write(encrypted_data)

print("File encrypted successfully!")

# Step 6: Protected file sharing
print("\nEncrypted file is ready for sharing.")
print("Share encrypted_file.txt securely.")
print("Share secret.key separately with the receiver.")

# Step 7: Load secret key for receiver
with open("secret.key", "rb") as key_file:
    receiver_key = key_file.read()

receiver_cipher = Fernet(receiver_key)

# Step 8: Decrypt received file
with open("encrypted_file.txt", "rb") as file:
    encrypted_content = file.read()

decrypted_data = receiver_cipher.decrypt(encrypted_content)

with open("decrypted_file.txt", "wb") as file:
    file.write(decrypted_data)

print("\nFile decrypted successfully!")

# Step 9: Display decrypted content
with open("decrypted_file.txt", "r") as file:
    print("\nDecrypted File Content:")
    print(file.read())

# Step 10: Display generated files
print("\nGenerated Files:")
for name in ["secret.key", "sample.txt",
             "encrypted_file.txt", "decrypted_file.txt"]:
    print(name, ":", "Created" if os.path.exists(name) else "Not Found")



In [ ]:
!pip install pandas

In [ ]:
#15

import pandas as pd

# Step 1: Create sample dataset
data = {
    "Name": ["Rahul", "Priya", "Amit", "Neha", "Karan"],
    "Age": [22, 25, 30, 28, 35],
    "Email": [
        "rahul@gmail.com",
        "priya@gmail.com",
        "amit@gmail.com",
        "neha@gmail.com",
        "karan@gmail.com"
    ],
    "Phone": [
        "9876543210",
        "8765432109",
        "7654321098",
        "9876501234",
        "8765401234"
    ],
    "City": [
        "Ahmedabad", "Surat", "Mumbai",
        "Ahmedabad", "Surat"
    ]
}

df = pd.DataFrame(data)

# Step 2: Display original dataset
print("Original Dataset:")
print(df)

# Step 3: Perform data anonymization
anonymous_df = df.copy()

# Replace names with anonymous IDs
anonymous_df["Name"] = [
    "User_" + str(i)
    for i in range(1, len(df) + 1)
]

# Generalize age into age groups
anonymous_df["Age"] = pd.cut(
    anonymous_df["Age"],
    bins=[0, 25, 35, 100],
    labels=["18-25", "26-35", "36+"],
    include_lowest=True
)

# Mask email addresses
anonymous_df["Email"] = "hidden@email.com"

# Mask phone numbers
anonymous_df["Phone"] = "XXXXXX" + df["Phone"].str[-4:]

# Step 4: Display anonymized dataset
print("\nAnonymized Dataset:")
print(anonymous_df)

# Step 5: Save anonymized dataset
anonymous_df.to_csv("anonymized_data.csv", index=False)

print("\nAnonymized dataset saved successfully!")
